# Лабораторная работа 5. Классификация отзывов нейронными сетями

В этой работе используется тот же датасет отзывов Kinopoisk, что и в лабораторной работе 4, но классические модели заменены на нейронные сети. Основная цель — построить полный NLP-пайплайн: от анализа и предобработки текста до обучения `Embedding + Conv1D` и `Embedding + BiGRU`, сравнения качества и проверки модели на новых отзывах.

## Содержание

1. Окружение и настройки  
2. Загрузка датасета  
3. Разведочный анализ данных  
4. Предобработка текста  
5. Токенизация и подготовка батчей  
6. Нейросетевые модели  
7. Обучение и сравнение качества  
8. Анализ ошибок  
9. Проверка на новых текстах  
10. Выводы

## 1. Окружение и настройки

Импортируем библиотеки, фиксируем seed, выбираем устройство для PyTorch и задаём основные константы. В окружении проекта уже есть PyTorch, поэтому лабораторная реализована на нём без добавления TensorFlow-зависимостей.

Импортируем библиотеки и задаём воспроизводимые параметры эксперимента: размеры выборок, словаря, последовательностей и обучения.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import spacy
import torch
import torch.nn as nn
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from spacy.lang.ru.stop_words import STOP_WORDS as SPACY_RU_STOPWORDS
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from wordcloud import WordCloud

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)
px.defaults.template = "plotly_white"

SEED = 42
LABEL_ORDER = ["negative", "neutral", "positive"]
LABEL_RU = {
    "negative": "негативный",
    "neutral": "нейтральный",
    "positive": "позитивный",
}

SAMPLES_PER_CLASS = 15_000
VALID_SIZE = 0.15
MAX_VOCAB_SIZE = 50_000
MIN_FREQ = 2
MAX_LEN_CAP = 400
MIN_LEN = 80
BATCH_SIZE = 128
MAX_EPOCHS = 6
PATIENCE = 2
LEARNING_RATE = 1e-3
EMBEDDING_DIM = 128
DROPOUT = 0.4

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

PREPROCESSING_VERSION = "lab5_pytorch_spacy_v1"
SPACY_MODEL = "ru_core_news_sm"
PREPARED_DATA_FILENAME = "kinopoisk_reviews_prepared.csv"


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(SEED)
DEVICE = resolve_device()
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

## 2. Загрузка датасета

Работаем с подготовленным CSV. Если файл отсутствует в папке пятой лабораторной, ячейка попытается создать hardlink из `LAB_IV/data/dataset`, а если файловая система не позволит — обычную копию.

Находим корень проекта и гарантируем, что подготовленный CSV доступен в папке `LAB_V/data/dataset` через hardlink или копирование.

In [ ]:
def find_project_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "LAB_V").exists() and (candidate / "LAB_IV").exists():
            return candidate
        if candidate.name == "LAB_V" and (candidate.parent / "LAB_IV").exists():
            return candidate.parent
    raise FileNotFoundError("Не удалось найти корень проекта с папками LAB_IV и LAB_V.")


PROJECT_DIR = find_project_dir()
LAB_V_DIR = PROJECT_DIR / "LAB_V"
LAB_IV_DIR = PROJECT_DIR / "LAB_IV"
DATA_DIR = LAB_V_DIR / "data"
DATASET_DIR = DATA_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_DIR / PREPARED_DATA_FILENAME
LAB4_DATA_PATH = LAB_IV_DIR / "data" / "dataset" / PREPARED_DATA_FILENAME


def ensure_prepared_dataset() -> Path:
    if DATA_PATH.exists():
        return DATA_PATH
    if not LAB4_DATA_PATH.exists():
        raise FileNotFoundError(
            "Подготовленный CSV не найден ни в LAB_V/data/dataset, ни в LAB_IV/data/dataset."
        )
    try:
        os.link(LAB4_DATA_PATH, DATA_PATH)
        print(f"Created hardlink: {DATA_PATH}")
    except OSError:
        shutil.copy2(LAB4_DATA_PATH, DATA_PATH)
        print(f"Copied dataset: {DATA_PATH}")
    return DATA_PATH


DATA_PATH = ensure_prepared_dataset()
print(f"Project dir: {PROJECT_DIR}")
print(f"Prepared data path: {DATA_PATH}")
print(f"File size: {DATA_PATH.stat().st_size / 1024**2:.1f} MB")

Загружаем таблицу отзывов, оставляем нужные колонки и проверяем базовую структуру.

In [ ]:
USE_COLUMNS = ["review", "sentiment", "source_label_dir", "source_file", "movie_id", "review_id"]
df = pd.read_csv(DATA_PATH, usecols=USE_COLUMNS)
df = df.dropna(subset=["review", "sentiment"]).copy()
df["review"] = df["review"].astype(str)
df["sentiment"] = df["sentiment"].astype(str)
df = df[df["sentiment"].isin(LABEL_ORDER)].reset_index(drop=True)
df["row_uid"] = df["source_file"].fillna("unknown").astype(str) + "::" + df.index.astype(str)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
display(df.head())
display(df.info(memory_usage="deep"))

## 3. Разведочный анализ данных

Проверим баланс классов, размеры отзывов, количество слов и уникальных слов. Эти статистики нужны не только для понимания датасета, но и для выбора `MAX_LEN` при padding последовательностей.

Считаем базовые признаки отзывов для EDA: длины в символах и словах, а также простые токены для частотного анализа.

In [ ]:
def tokenize_for_eda(text: str) -> list[str]:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"[^а-яa-z\s]", " ", text)
    return [token for token in text.split() if len(token) > 1]


df["raw_word_count"] = df["review"].str.split().map(len)
df["raw_char_count"] = df["review"].str.len()
df["raw_unique_word_count"] = df["review"].map(lambda text: len(set(str(text).lower().split())))

class_distribution = (
    df["sentiment"]
    .value_counts()
    .reindex(LABEL_ORDER)
    .rename_axis("sentiment")
    .reset_index(name="count")
)
class_distribution["share"] = class_distribution["count"] / class_distribution["count"].sum()

display(class_distribution)

display(
    df.groupby("sentiment")[["raw_word_count", "raw_unique_word_count", "raw_char_count"]]
    .agg(["mean", "median", "min", "max"])
    .round(2)
)

overall_length_stats = df["raw_word_count"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2)
display(overall_length_stats.to_frame("raw_word_count"))

Визуализируем распределение классов и длин отзывов. Длинный правый хвост ожидаем для рецензий: часть текстов намного длиннее типичного отзыва.

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Распределение классов", "Длина отзывов в словах"),
)
fig.add_trace(
    go.Bar(
        x=class_distribution["sentiment"],
        y=class_distribution["count"],
        text=class_distribution["share"].map(lambda value: f"{value:.1%}"),
        marker_color=["#c44e52", "#8172b2", "#55a868"],
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=df["raw_word_count"],
        nbinsx=80,
        marker_color="#4c78a8",
        opacity=0.85,
    ),
    row=1,
    col=2,
)
fig.update_layout(height=420, showlegend=False, title_text="Базовый EDA")
fig.update_xaxes(title_text="Класс", row=1, col=1)
fig.update_yaxes(title_text="Количество", row=1, col=1)
fig.update_xaxes(title_text="Слова", row=1, col=2)
fig.update_yaxes(title_text="Количество отзывов", row=1, col=2)
fig.show()

Посмотрим на самые частые слова по классам по всему датасету. Это часть EDA, поэтому здесь не используем обучающий subset и не подглядываем в train/test split.

In [ ]:
EDA_STOPWORDS = set(SPACY_RU_STOPWORDS) | {"это", "все", "ещё", "еще", "который", "фильм"}

class_word_counters: dict[str, Counter] = {}
for label in LABEL_ORDER:
    counter = Counter()
    for text in df.loc[df["sentiment"] == label, "review"]:
        counter.update(
            token
            for token in tokenize_for_eda(text)
            if token not in EDA_STOPWORDS and len(token) >= 3
        )
    class_word_counters[label] = counter

top_words_rows = []
for label, counter in class_word_counters.items():
    for word, count in counter.most_common(20):
        top_words_rows.append({"sentiment": label, "word": word, "count": count})

top_words_df = pd.DataFrame(top_words_rows)
display(top_words_df.head(15))

fig = px.bar(
    top_words_df,
    x="count",
    y="word",
    color="sentiment",
    facet_col="sentiment",
    orientation="h",
    category_orders={"sentiment": LABEL_ORDER},
    height=520,
    title="Топ-20 слов по классам",
)
fig.update_yaxes(matches=None, autorange="reversed")
fig.update_layout(showlegend=False)
fig.show()

WordCloud помогает быстро увидеть лексические маркеры классов. В отличие от таблицы частот, здесь легче заметить повторяющиеся смысловые зоны.

In [ ]:
FONT_CANDIDATES = [
    Path("/System/Library/Fonts/Supplemental/Arial Unicode.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial.ttf"),
    Path("/System/Library/Fonts/Helvetica.ttc"),
    Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
    Path("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"),
]
WORDCLOUD_FONT_PATH = next((str(path) for path in FONT_CANDIDATES if path.exists()), None)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, label in zip(axes, LABEL_ORDER):
    text = " ".join(
        word
        for word, _ in class_word_counters[label].most_common(3_000)
    )
    wc = WordCloud(
        width=900,
        height=500,
        background_color="white",
        colormap="viridis",
        collocations=False,
        font_path=WORDCLOUD_FONT_PATH,
        random_state=SEED,
    ).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{label} / {LABEL_RU[label]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Предобработка текста

Для нейронной сети текст приводится к нижнему регистру, очищается от пунктуации и символов, затем лемматизируется через `ru_core_news_sm`. Стоп-слова удаляются, но отрицания `не` и `нет` сохраняются, потому что они важны для тональности.

Загружаем русскую модель spaCy и настраиваем стоп-слова так, чтобы отрицания `не` и `нет` оставались в тексте.

In [ ]:
try:
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
except OSError as exc:
    raise RuntimeError(
        "Не найдена модель ru_core_news_sm. Установите её командой: "
        "uv run python -m spacy download ru_core_news_sm"
    ) from exc

STOPWORDS = set(SPACY_RU_STOPWORDS)
STOPWORDS.discard("не")
STOPWORDS.discard("нет")

print(f"spaCy model: {SPACY_MODEL}")
print(f"Stopwords: {len(STOPWORDS)}")
print(f"'не' in STOPWORDS: {'не' in STOPWORDS}")
print(f"'нет' in STOPWORDS: {'нет' in STOPWORDS}")

Функции ниже используются и для обучающего датасета, и для новых пользовательских текстов. Это важно: инференс должен проходить через точно такой же preprocessing, что и обучение.

In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"[^а-яa-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_doc(doc) -> str:
    lemmas = []
    for token in doc:
        lemma = token.lemma_.lower().strip().replace("ё", "е")
        if not lemma or lemma in STOPWORDS or len(lemma) < 2 or token.is_space:
            continue
        lemmas.append(lemma)
    return " ".join(lemmas)


def preprocess_text(text: str) -> str:
    return clean_doc(nlp(normalize_text(text)))


def preprocess_texts(texts: pd.Series | list[str], batch_size: int = 256, n_process: int = 1) -> list[str]:
    normalized = [normalize_text(text) for text in texts]
    cleaned = []
    docs = nlp.pipe(normalized, batch_size=batch_size, n_process=n_process)
    for doc in tqdm(docs, total=len(normalized), desc="spaCy preprocessing"):
        cleaned.append(clean_doc(doc))
    return cleaned


example_text = df.loc[0, "review"]
print("До:", example_text[:350])
print("После:", preprocess_text(example_text)[:350])

Перед обучением формируем практичную стратифицированную выборку. Полный датасет используется для EDA, а нейросети обучаются на сбалансированной части, чтобы запуск лабораторной оставался реалистичным по времени.

In [ ]:
def sample_by_class(
    frame: pd.DataFrame,
    samples_per_class: int,
    label_col: str = "sentiment",
    random_state: int = SEED,
) -> pd.DataFrame:
    """Return a balanced stratified subset with exactly N rows per class."""
    class_counts = frame[label_col].value_counts()
    too_small = {
        label: int(class_counts.get(label, 0))
        for label in LABEL_ORDER
        if class_counts.get(label, 0) < samples_per_class
    }
    if too_small:
        raise ValueError(
            f"Недостаточно строк для SAMPLES_PER_CLASS={samples_per_class}: {too_small}"
        )

    sampled_parts = [
        frame[frame[label_col] == label].sample(
            n=samples_per_class,
            random_state=random_state,
        )
        for label in LABEL_ORDER
    ]
    return (
        pd.concat(sampled_parts, ignore_index=True)
        .sample(frac=1.0, random_state=random_state)
        .reset_index(drop=True)
    )


train_full_raw, test_raw = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["sentiment"],
)

train_pool_raw = sample_by_class(train_full_raw, SAMPLES_PER_CLASS)
train_pool_raw["split"] = "train_pool"
test_raw = test_raw.copy()
test_raw["split"] = "test"
model_raw_df = pd.concat([train_pool_raw, test_raw], ignore_index=True)

print("Full train rows before subset:", len(train_full_raw))
print("Modeling rows:", len(model_raw_df))
display(
    pd.DataFrame(
        [
            {"part": "train_full_before_subset", "rows": len(train_full_raw)},
            {"part": "train_pool_balanced_subset", "rows": len(train_pool_raw)},
            {"part": "test_full_holdout", "rows": len(test_raw)},
        ]
    )
)
display(model_raw_df["split"].value_counts().to_frame("count"))
display(pd.crosstab(model_raw_df["split"], model_raw_df["sentiment"]))

Кэшируем очищенные тексты в parquet. Повторный запуск ноутбука не будет заново лемматизировать те же отзывы, если версия preprocessing и состав строк совпадают.

In [ ]:
CACHE_PATH = DATASET_DIR / "kinopoisk_reviews_cleaned.parquet"
CACHE_META_PATH = DATASET_DIR / "kinopoisk_reviews_cleaned.meta.json"


def row_uid_hash(row_uids: pd.Series) -> str:
    joined = "\n".join(row_uids.astype(str).tolist()).encode("utf-8")
    return hashlib.sha256(joined).hexdigest()


def expected_cache_meta(raw_frame: pd.DataFrame) -> dict:
    return {
        "version": PREPROCESSING_VERSION,
        "rows": int(len(raw_frame)),
        "row_uid_hash": row_uid_hash(raw_frame["row_uid"]),
        "spacy_model": SPACY_MODEL,
        "samples_per_class": SAMPLES_PER_CLASS,
        "test_split": "full_20_percent_stratified_holdout",
    }


def cache_is_valid(raw_frame: pd.DataFrame) -> bool:
    if not CACHE_PATH.exists() or not CACHE_META_PATH.exists():
        return False
    try:
        actual = json.loads(CACHE_META_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    return actual == expected_cache_meta(raw_frame)


if cache_is_valid(model_raw_df):
    model_df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded cached preprocessing: {CACHE_PATH}")
else:
    model_df = model_raw_df.copy()
    model_df["cleaned_text"] = preprocess_texts(model_df["review"], batch_size=256, n_process=1)
    model_df.to_parquet(CACHE_PATH, index=False)
    CACHE_META_PATH.write_text(
        json.dumps(expected_cache_meta(model_raw_df), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved preprocessing cache: {CACHE_PATH}")

model_df["cleaned_word_count"] = model_df["cleaned_text"].str.split().map(len)
model_df = model_df[model_df["cleaned_word_count"] > 0].reset_index(drop=True)

display(model_df[["review", "cleaned_text", "sentiment", "split"]].head())
display(model_df.groupby(["split", "sentiment"])["cleaned_word_count"].agg(["count", "mean", "median"]).round(2))

## 5. Токенизация и подготовка батчей

Словарь строится только по обучающей части, чтобы не подглядывать в validation/test. Последовательности дополняются `PAD`-токеном до выбранной длины.

После кэшированной предобработки отделяем train, validation и test; длину последовательности выбираем по 95-му перцентилю train-текстов.

In [ ]:
train_pool_df = model_df[model_df["split"] == "train_pool"].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

train_df, val_df = train_test_split(
    train_pool_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=train_pool_df["sentiment"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_lengths = train_df["cleaned_text"].str.split().map(len)
p95_len = int(np.percentile(train_lengths, 95))
MAX_LEN = int(min(MAX_LEN_CAP, max(MIN_LEN, p95_len)))

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"95th percentile train length: {p95_len}")
print(f"MAX_LEN: {MAX_LEN}")

display(
    pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "rows": [len(train_df), len(val_df), len(test_df)],
        }
    )
)

Строим словарь только по train subset, фиксируя `PAD_IDX=0`, `UNK_IDX=1`, `MAX_VOCAB_SIZE=50_000` и `MIN_FREQ=2`.

In [ ]:
def iter_tokens(texts: pd.Series):
    for text in texts:
        yield from str(text).split()


def build_vocab(texts: pd.Series, max_vocab_size: int = MAX_VOCAB_SIZE, min_freq: int = MIN_FREQ):
    counter = Counter(iter_tokens(texts))
    words = [word for word, count in counter.most_common(max_vocab_size - 2) if count >= min_freq]
    word_to_idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    word_to_idx.update({word: idx for idx, word in enumerate(words, start=2)})
    return word_to_idx, counter


def encode_text(text: str, word_to_idx: dict[str, int], max_len: int = MAX_LEN) -> list[int]:
    token_ids = [word_to_idx.get(token, UNK_IDX) for token in str(text).split()[:max_len]]
    if len(token_ids) < max_len:
        token_ids.extend([PAD_IDX] * (max_len - len(token_ids)))
    return token_ids


def encode_texts(texts: pd.Series | list[str], word_to_idx: dict[str, int], max_len: int = MAX_LEN) -> np.ndarray:
    return np.asarray([encode_text(text, word_to_idx, max_len) for text in texts], dtype=np.int64)


word_to_idx, token_counter = build_vocab(train_df["cleaned_text"])
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
label_to_id = {label: idx for idx, label in enumerate(LABEL_ORDER)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print(f"Vocabulary size: {len(word_to_idx):,}")
print("Most common tokens:", token_counter.most_common(15))
print("Labels:", label_to_id)

Кодируем очищенные тексты в массивы одинаковой длины и переводим метки классов в числовые идентификаторы.

In [ ]:
X_train = encode_texts(train_df["cleaned_text"], word_to_idx)
X_val = encode_texts(val_df["cleaned_text"], word_to_idx)
X_test = encode_texts(test_df["cleaned_text"], word_to_idx)

y_train = train_df["sentiment"].map(label_to_id).to_numpy(dtype=np.int64)
y_val = val_df["sentiment"].map(label_to_id).to_numpy(dtype=np.int64)
y_test = test_df["sentiment"].map(label_to_id).to_numpy(dtype=np.int64)

print(X_train.shape, X_val.shape, X_test.shape)

Оборачиваем массивы в `Dataset` и `DataLoader`, чтобы один и тот же pipeline работал для обучения, validation, test и новых текстов.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences: np.ndarray, labels: np.ndarray):
        self.sequences = torch.as_tensor(sequences, dtype=torch.long)
        self.labels = torch.as_tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return self.sequences[idx], self.labels[idx]


def make_loader(sequences: np.ndarray, labels: np.ndarray, batch_size: int = BATCH_SIZE, shuffle: bool = False) -> DataLoader:
    return DataLoader(
        ReviewDataset(sequences, labels),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )


train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 6. Нейросетевые модели

Обучим две архитектуры: сверточную модель, которая хорошо ловит локальные n-граммы, и двунаправленную GRU, которая учитывает порядок токенов в обе стороны.

Описываем две нейросетевые архитектуры из задания: сверточную `TextCNN` и рекуррентную `TextBiGRU`.

In [ ]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = EMBEDDING_DIM,
        num_filters: int = 128,
        kernel_size: int = 5,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=num_filters,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
        )
        self.activation = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x).transpose(1, 2)
        features = self.pool(self.activation(self.conv(embedded))).squeeze(-1)
        return self.classifier(self.dropout(features))


class TextBiGRU(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = EMBEDDING_DIM,
        hidden_size: int = 128,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(self.dropout(features))


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


model_factories = {
    "TextCNN": lambda: TextCNN(len(word_to_idx), len(LABEL_ORDER)),
    "TextBiGRU": lambda: TextBiGRU(len(word_to_idx), len(LABEL_ORDER)),
}

for model_name, factory in model_factories.items():
    model = factory()
    print(f"{model_name}: {count_parameters(model):,} trainable parameters")

## 7. Обучение и сравнение качества

Модели обучаются с `CrossEntropyLoss`, оптимизатором Adam и ранней остановкой по `macro-F1` на validation. `macro-F1` важен, потому что классы в исходном датасете несбалансированы.

Функции ниже отвечают за метрики, один проход по данным, оценку модели и обучение с early stopping по validation macro-F1.

In [ ]:
def labels_from_ids(label_ids: np.ndarray | list[int]) -> list[str]:
    return [id_to_label[int(label_id)] for label_id in label_ids]


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer=None) -> dict:
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    y_true_batches = []
    y_pred_batches = []

    for sequences, labels in loader:
        sequences = sequences.to(DEVICE)
        labels = labels.to(DEVICE)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(sequences)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        y_true_batches.append(labels.detach().cpu().numpy())
        y_pred_batches.append(logits.argmax(dim=1).detach().cpu().numpy())

    y_true = np.concatenate(y_true_batches)
    y_pred = np.concatenate(y_pred_batches)
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


@torch.inference_mode()
def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    y_true_batches = []
    y_pred_batches = []
    prob_batches = []

    for sequences, labels in loader:
        sequences = sequences.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(sequences)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        probabilities = torch.softmax(logits, dim=1)
        y_true_batches.append(labels.detach().cpu().numpy())
        y_pred_batches.append(logits.argmax(dim=1).detach().cpu().numpy())
        prob_batches.append(probabilities.detach().cpu().numpy())

    y_true = np.concatenate(y_true_batches)
    y_pred = np.concatenate(y_pred_batches)
    probabilities = np.vstack(prob_batches)
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(loader.dataset)
    return {
        "metrics": metrics,
        "y_true": y_true,
        "y_pred": y_pred,
        "probabilities": probabilities,
    }


def train_model(model_name: str, model: nn.Module) -> tuple[nn.Module, pd.DataFrame, dict]:
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_state = copy.deepcopy(model.state_dict())
    best_macro_f1 = -math.inf
    epochs_without_improvement = 0
    history = []
    started_at = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None)

        row = {
            "model": model_name,
            "epoch": epoch,
            **{f"train_{key}": value for key, value in train_metrics.items()},
            **{f"val_{key}": value for key, value in val_metrics.items()},
        }
        history.append(row)
        print(
            f"{model_name} | epoch {epoch:02d} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1 + 1e-4:
            best_macro_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping for {model_name} at epoch {epoch}")
                break

    elapsed = time.perf_counter() - started_at
    model.load_state_dict(best_state)
    best_info = {
        "model": model_name,
        "best_val_macro_f1": best_macro_f1,
        "training_time_sec": elapsed,
        "params": count_parameters(model),
    }
    return model, pd.DataFrame(history), best_info

Обучаем обе архитектуры на одинаковых данных и собираем историю validation-метрик для сравнения.

In [ ]:
trained_models = {}
history_frames = []
validation_summaries = []

for model_name, factory in model_factories.items():
    print("=" * 80)
    print(f"Training {model_name}")
    set_seed(SEED)
    model, history_df, best_info = train_model(model_name, factory())
    trained_models[model_name] = model
    history_frames.append(history_df)
    validation_summaries.append(best_info)

history_df = pd.concat(history_frames, ignore_index=True)
validation_summary_df = pd.DataFrame(validation_summaries).sort_values("best_val_macro_f1", ascending=False)
display(validation_summary_df)

Визуализируем динамику validation macro-F1 и loss, чтобы увидеть, на каких эпохах модели перестают улучшаться.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Validation macro-F1", "Validation loss"))
for model_name in history_df["model"].unique():
    part = history_df[history_df["model"] == model_name]
    fig.add_trace(go.Scatter(x=part["epoch"], y=part["val_macro_f1"], mode="lines+markers", name=model_name), row=1, col=1)
    fig.add_trace(go.Scatter(x=part["epoch"], y=part["val_loss"], mode="lines+markers", name=model_name, showlegend=False), row=1, col=2)
fig.update_layout(height=420, title_text="Динамика обучения")
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Macro-F1", row=1, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=2)
fig.show()

Проверяем обе модели на отложенной test-выборке и выбираем лучшую по `macro-F1`.

In [ ]:
test_predictions = {}
test_rows = []

for model_name, model in trained_models.items():
    result = evaluate_model(model.to(DEVICE), test_loader)
    test_predictions[model_name] = result
    row = {
        "model": model_name,
        **result["metrics"],
        "params": count_parameters(model),
        "training_time_sec": validation_summary_df.loc[
            validation_summary_df["model"] == model_name,
            "training_time_sec",
        ].iloc[0],
    }
    test_rows.append(row)

results_df = pd.DataFrame(test_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]
best_result = test_predictions[best_model_name]

display(results_df)
print(f"Best model by test macro-F1: {best_model_name}")

Для лучшей модели печатаем `classification_report` и строим confusion matrix по полной test-выборке.

In [ ]:
y_true_labels = labels_from_ids(best_result["y_true"])
y_pred_labels = labels_from_ids(best_result["y_pred"])

print(classification_report(y_true_labels, y_pred_labels, labels=LABEL_ORDER, target_names=LABEL_ORDER, digits=4))

cm = confusion_matrix(y_true_labels, y_pred_labels, labels=LABEL_ORDER)
cm_df = pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER)
fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    labels={"x": "Предсказание", "y": "Истинный класс", "color": "Количество"},
    title=f"Confusion matrix: {best_model_name}",
)
fig.update_layout(height=500)
fig.show()

## 8. Анализ ошибок

Ниже несколько ошибочных предсказаний лучшей модели. Для каждого примера добавлена уверенность модели и короткая эвристическая подсказка: совпадают ли слова отзыва с частотными словами истинного или предсказанного класса.

Выбираем 3-5 ошибочных примеров и добавляем короткий диагностический комментарий к каждой ошибке.

In [ ]:
def shorten_text(text: str, max_chars: int = 420) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= max_chars else text[: max_chars - 1] + "…"


class_top_terms = {
    label: {word for word, _ in counter.most_common(80)}
    for label, counter in class_word_counters.items()
}

analysis_df = test_df.copy()
analysis_df["true_label"] = y_true_labels
analysis_df["pred_label"] = y_pred_labels
analysis_df["pred_confidence"] = best_result["probabilities"].max(axis=1)
analysis_df["review_short"] = analysis_df["review"].map(shorten_text)
analysis_df["cleaned_short"] = analysis_df["cleaned_text"].map(lambda text: shorten_text(text, 240))

errors_df = analysis_df[analysis_df["true_label"] != analysis_df["pred_label"]].copy()


def explain_error(row: pd.Series) -> str:
    tokens = set(str(row["cleaned_text"]).split())
    true_overlap = tokens & class_top_terms.get(row["true_label"], set())
    pred_overlap = tokens & class_top_terms.get(row["pred_label"], set())
    if len(pred_overlap) > len(true_overlap):
        return "В тексте больше частотных маркеров предсказанного класса, чем истинного."
    if row["pred_confidence"] < 0.55:
        return "Модель неуверенна: вероятности классов близки, отзыв находится около границы решения."
    if len(tokens) < 20:
        return "Короткий очищенный текст: после preprocessing осталось мало сигналов для класса."
    return "Сигналы классов смешаны: вероятно, отзыв содержит и похвалу, и критику."


if errors_df.empty:
    display(Markdown("Ошибок на test-выборке не найдено. Это редкая, но приятная ситуация."))
else:
    error_examples = errors_df.sort_values("pred_confidence", ascending=False).head(5).copy()
    error_examples["comment"] = error_examples.apply(explain_error, axis=1)
    display(
        error_examples[
            ["true_label", "pred_label", "pred_confidence", "review_short", "cleaned_short", "comment"]
        ].style.format({"pred_confidence": "{:.3f}"})
    )

## 9. Проверка на новых текстах

Создадим пять коротких отзывов, пропустим их через тот же preprocessing, тот же словарь и лучшую модель.

Проверяем лучшую модель на пяти новых коротких отзывах с ожидаемыми метками и вероятностями по всем классам.

In [ ]:
new_examples = [
    {
        "review": "Фильм удивительно теплый: актеры играют живо, финал трогает, а история долго не отпускает.",
        "expected_label": "positive",
    },
    {
        "review": "Сценарий разваливается на глазах, диалоги деревянные, а два часа кажутся бесконечными.",
        "expected_label": "negative",
    },
    {
        "review": "Обычная картина на вечер: есть несколько удачных сцен, но ничего принципиально нового она не предлагает.",
        "expected_label": "neutral",
    },
    {
        "review": "Не ожидал такого сильного впечатления: режиссура уверенная, музыка точная, персонажам веришь.",
        "expected_label": "positive",
    },
    {
        "review": "Визуально красиво, но сюжет пустой, герои раздражают, пересматривать точно не хочется.",
        "expected_label": "negative",
    },
]

new_reviews_df = pd.DataFrame(new_examples)
new_cleaned = preprocess_texts(new_reviews_df["review"], batch_size=32, n_process=1)
new_sequences = encode_texts(new_cleaned, word_to_idx)
new_dummy_labels = np.zeros(len(new_sequences), dtype=np.int64)
new_loader = make_loader(new_sequences, new_dummy_labels, batch_size=len(new_sequences))

@torch.inference_mode()
def predict_texts(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    probability_batches = []
    prediction_batches = []
    for sequences, _ in loader:
        sequences = sequences.to(DEVICE)
        logits = model(sequences)
        probabilities = torch.softmax(logits, dim=1).detach().cpu().numpy()
        predictions = probabilities.argmax(axis=1)
        probability_batches.append(probabilities)
        prediction_batches.append(predictions)
    probabilities = np.vstack(probability_batches)
    predictions = np.concatenate(prediction_batches)
    result = pd.DataFrame(probabilities, columns=[f"p_{label}" for label in LABEL_ORDER])
    result.insert(0, "predicted_label", labels_from_ids(predictions))
    return result

new_predictions = predict_texts(best_model.to(DEVICE), new_loader)
new_result_df = new_reviews_df.assign(cleaned_text=new_cleaned).join(new_predictions)
display(new_result_df.style.format({column: "{:.3f}" for column in new_result_df.columns if column.startswith("p_")}))

## 10. Выводы

Итоговая ячейка формирует выводы по фактическим метрикам после запуска ноутбука.

Формируем итоговые выводы автоматически на основе реально полученных метрик после запуска ноутбука.

In [ ]:
best_row = results_df.iloc[0]
second_row = results_df.iloc[1] if len(results_df) > 1 else None

summary_lines = [
    "## Итоговые выводы",
    f"- Лучшей моделью по `macro-F1` на test-выборке стала `{best_row['model']}`: "
    f"macro-F1 = `{best_row['macro_f1']:.4f}`, accuracy = `{best_row['accuracy']:.4f}`.",
    f"- Для обучения использовалась сбалансированная практичная выборка: "
    f"`{SAMPLES_PER_CLASS}` отзывов на класс в train-pool; test-выборка оставлена полным стратифицированным 20% holdout.",
    f"- `MAX_LEN = {MAX_LEN}` выбран по 95-му перцентилю длин обучающих очищенных текстов с ограничением `MAX_LEN_CAP = {MAX_LEN_CAP}`.",
    "- Ошибки чаще возникают на смешанных или коротких отзывах, где после очистки остаётся мало однозначных маркеров тональности.",
    "- Для новых отзывов используется тот же preprocessing, словарь и padding, поэтому инференс согласован с обучением.",
]
if second_row is not None:
    diff = best_row["macro_f1"] - second_row["macro_f1"]
    summary_lines.insert(
        2,
        f"- Разница с `{second_row['model']}` по macro-F1 составила `{diff:.4f}`.",
    )

display(Markdown("\n".join(summary_lines)))